In [ ]:
# ========================= CDR Framework: Divide-&-Conquer Batching → Global CDR → Final Length-Extension =========================
# Non-overlapping batches; recursively re-batch the union if it stays > batch_size.

import time, random
import pandas as pd
import numpy as np

# ----------------------- Loader -----------------------
def load_ctx_drug(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, index_col=0)
    df.columns = df.columns.str.strip()
    df.index = df.index.astype(str).str.strip()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

# ----------------------- Pairwise tissue sets -----------------------
def pairwise_tissue_sets(df: pd.DataFrame):
    drugs = list(df.columns)
    tissues = list(df.index)
    n = len(drugs)
    S = [[set() for _ in range(n)] for __ in range(n)]
    vals = df.values  # m_tissues × n_drugs
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            a = vals[:, i]
            b = vals[:, j]
            mask = (~np.isnan(a)) & (~np.isnan(b)) & (a > b)
            if np.any(mask):
                S[i][j] = {tissues[k] for k in np.flatnonzero(mask)}
    return drugs, tissues, S

# ----------------------- CDR (chain-λ) on any subset -----------------------
def cdr_longest_path_chain_lambda(df_sub: pd.DataFrame, lam: int):
    drugs, tissues, S = pairwise_tissue_sets(df_sub)
    n = len(drugs)
    adj = {i: [j for j in range(n) if i != j and len(S[i][j]) > 0] for i in range(n)}
    best_path_idx, best_S = [], set()

    for start in range(n):
        stack = [(start, [start], None)]
        while stack:
            cur, path, S_common = stack.pop()
            for nxt in adj[cur]:
                if nxt in path:
                    continue
                Sab = S[cur][nxt]
                newS = Sab if S_common is None else (S_common & Sab)
                if len(newS) >= lam:
                    new_path = path + [nxt]
                    stack.append((nxt, new_path, newS))
                    if len(new_path) > len(best_path_idx):
                        best_path_idx, best_S = new_path, set(newS)

    return [drugs[i] for i in best_path_idx], best_S, []

# ----------------------- Make non-overlapping batches -----------------------
def make_batches(drug_list, batch_size=45, seed=42):
    rng = random.Random(seed)
    order = drug_list[:]
    rng.shuffle(order)
    return [order[i:i+batch_size] for i in range(0, len(order), batch_size)]

# ----------------------- Final length extension (global, exact tset) -----------------------
def extend_order_by_exact_tset_between_pairs(df: pd.DataFrame, order: list[str], tset: set[str], lam: int):
    if len(order) < 2 or not tset:
        return order[:]

    df_t = df.loc[list(tset), :]  # restrict to the exact tissues that supported the global path
    all_drugs = list(df_t.columns)
    idx_map = {d:i for i,d in enumerate(all_drugs)}
    _, _, S = pairwise_tissue_sets(df_t)

    cur = order[:]
    changed = True
    while changed:
        changed = False
        i = 0
        while i < len(cur) - 1:
            A, B = cur[i], cur[i+1]
            Ai, Bi = idx_map[A], idx_map[B]

            # candidates not currently inside the (i, i+1) gap
            candidates = [d for d in all_drugs if d not in cur or not (cur.index(d) >= i and cur.index(d) <= i+1)]

            insertables = []
            for X in candidates:
                Xi = idx_map[X]
                if not (set(S[Ai][Xi]) >= set(tset) and set(S[Xi][Bi]) >= set(tset)):
                    continue
                inter = set(S[Ai][Xi]) & set(S[Xi][Bi])
                if inter == set(tset):
                    insertables.append(X)

            if insertables:
                
                insertables.sort(key=lambda x: min(len(S[idx_map[A]][idx_map[x]]),
                                                   len(S[idx_map[x]][idx_map[B]])), reverse=True)
                for X in insertables:
                    if X in cur:
                        pos = cur.index(X)
                        if pos == i or pos == i+1:  # already in the gap
                            continue
                        cur.pop(pos)
                        if pos < i:
                            i -= 1
                    cur.insert(i+1, X)
                    changed = True
                i += len(insertables) + 1
            else:
                i += 1

    return cur

# ----------------------- One DC pass: batch → per-batch CDR → union -----------------------
def dc_pass_union(df: pd.DataFrame, work_drugs: list[str], lam: int, batch_size: int, seed: int, verbose: bool):
    batches = make_batches(work_drugs, batch_size=batch_size, seed=seed)
    if verbose:
        print(f"  DC pass on set={len(work_drugs)} → {len(batches)} batches (size≈{batch_size})")
    hc_union = set()
    batch_summ = []
    for bi, cols in enumerate(batches, 1):
        sub = df[cols]
        path, tset, _ = cdr_longest_path_chain_lambda(sub, lam)
        batch_summ.append({"batch": bi, "size": len(cols), "path_len": len(path), "tset_size": len(tset)})
        if tset and path:
            hc_union.update(path)
    if verbose:
        print(f"  → union after pass: {len(hc_union)} drugs (non-empty paths only)")
    return list(hc_union), batches, batch_summ

# ----------------------- Top-level: Divide-&-Conquer framework -----------------------
def run_framework_divide_conquer(
    csv_path: str,
    lam: int = 30,
    batch_size: int = 45,
    seed: int = 42,
    max_passes: int = 5,
    verbose: bool = True
):
    """
    Repeatedly:
      - split current set into disjoint batches
      - run CDR per batch
      - union drugs that appear in non-empty batch paths
    until the working set size ≤ batch_size OR no further shrinkage OR max_passes reached.

    Then:
      - run ONE global CDR on the final working set
      - do global length-extension with EXACT global tissue set
    """
    t0 = time.time()
    df = load_ctx_drug(csv_path)
    all_drugs = list(df.columns)

    history = []
    work_set = all_drugs[:]
    prev_size = len(work_set)

    if verbose:
        print(f"Start DC framework | total drugs={prev_size}, batch_size={batch_size}, λ={lam}")

    for p in range(1, max_passes+1):
        if verbose: print(f"\n[Pass {p}] working set = {len(work_set)} drugs")
        hc_union, batches, batch_summ = dc_pass_union(df, work_set, lam, batch_size, seed + p - 1, verbose)
        history.append({"pass": p, "batches": batches, "batch_summary": batch_summ, "union_size": len(hc_union)})

        if len(hc_union) == 0:
            if verbose: print("  → union is empty; stopping.")
            work_set = []
            break

        if len(hc_union) <= batch_size:
            if verbose: print(f"  → union ≤ batch_size ({len(hc_union)} ≤ {batch_size}); stopping passes.")
            work_set = hc_union
            break

        if len(hc_union) >= prev_size:
            if verbose: print(f"  → no shrinkage ({len(hc_union)} ≥ {prev_size}); stopping to avoid loop.")
            work_set = hc_union
            break

        # continue with smaller set
        prev_size = len(hc_union)
        work_set = hc_union

    # Global stage
    if len(work_set) < 2:
        total_time = time.time() - t0
        return {
            "passes": history,
            "working_set": work_set,
            "global_order": work_set[:],
            "global_tissues": set(),
            "final_order": work_set[:],
            "runtime_s": total_time
        }

    if verbose:
        print(f"\nGlobal stage on {len(work_set)} drugs")

    df_work = df[work_set]
    global_order, S_global, _ = cdr_longest_path_chain_lambda(df_work, lam)
    if verbose:
        print("  Global CDR order:", " > ".join(global_order) if global_order else "(empty)")
        print(f"  |S_global| = {len(S_global)}")

    final_order = extend_order_by_exact_tset_between_pairs(df, global_order, S_global, lam)
    if verbose:
        print("  Final length-extended order:")
        print("   ", " > ".join(final_order) if final_order else "(empty)")

    total_time = time.time() - t0
    return {
        "passes": history,
        "working_set": work_set,
        "global_order": global_order,
        "global_tissues": S_global,
        "final_order": final_order,
        "runtime_s": total_time
    }


In [ ]:
 csv = "efficacy.csv"
 out = run_framework_divide_conquer(csv, lam=30, batch_size=45, seed=42, max_passes=6, verbose=True)
 print(f"\nTotal runtime: {out['runtime_s']:.2f} s")
 # Inspect:
 out["passes"][-1]["batch_summary"]
 out["working_set"], out["global_order"], out["final_order"]
